# 05 — DICOM header passHeader-only sweep of the competition data. No pixels are decoded, so this is I/Oon one file per series rather than per slice, and it should finish in minutes.It produces two files, each fixing one weakness in the paper:| Output | Fixes || --- | --- || `series_headers.csv` | Section 8's missing field-of-view distribution || `study_sites.csv` | Section 7's randomly stratified folds, which leak by site |Attach the competition dataset to this notebook before running. GPU is not needed.

## 1. Find the dataThe competition path has a `competitions/` segment that is easy to guess wrong,so print the tree rather than assuming it.

In [ ]:
import osdef peek(path, depth=0, maxdepth=4, width=6):    if depth > maxdepth:        return    try:        entries = sorted(os.listdir(path))    except Exception as exc:        print('  ' * depth + f'[{exc.__class__.__name__}]')        return    name = os.path.basename(path) or path    print('  ' * depth + name + f'/   ({len(entries)} entries)')    for e in entries[:width]:        p = os.path.join(path, e)        if os.path.isdir(p):            peek(p, depth + 1, maxdepth, width)        else:            print('  ' * (depth + 1) + e)    if len(entries) > width:        print('  ' * (depth + 1) + f'... {len(entries) - width} more')peek('/kaggle/input', maxdepth=4)

## 2. Point at the training imagesCopy the directory of per-study folders from the tree above into `TRAIN_DIR`.It is the one whose children are long UID-looking names.

In [ ]:
TRAIN_DIR = '/kaggle/input/CHANGE/ME/train_images'   # <-- edit this lineimport osassert os.path.isdir(TRAIN_DIR), f'not a directory: {TRAIN_DIR}'studies = [e.name for e in os.scandir(TRAIN_DIR) if e.is_dir()]print(f'{len(studies):,} studies')assert len(studies) > 1000, 'expected about 4,407 study folders - wrong directory?'

## 3. One file per series`scandir` stops at the first file in each series folder, so this listsdirectories rather than walking every slice. The earlier pipeline hung for halfan hour on a recursive glob; this is the reason for the care.

In [ ]:
import timet0 = time.time()tasks = []for s in studies:    for se in os.scandir(os.path.join(TRAIN_DIR, s)):        if not se.is_dir():            continue        first = next((f.path for f in os.scandir(se.path) if f.is_file()), None)        if first:            tasks.append((s, se.name, first))print(f'{len(tasks):,} series found in {time.time() - t0:.0f}s')

## 4. Read the headers`stop_before_pixels=True` is what makes this cheap. The tags below are constantwithin a series, so one file per series is enough.

In [ ]:
import pydicom, pandas as pdfrom concurrent.futures import ThreadPoolExecutordef read_one(task):    study, series, path = task    row = {'StudyInstanceUID': study, 'SeriesInstanceUID': series}    try:        h = pydicom.dcmread(path, stop_before_pixels=True, force=True)    except Exception as exc:        row['error'] = f'{exc.__class__.__name__}: {exc}'[:100]        return row    for key, tag in (('Manufacturer', 'Manufacturer'), ('Model', 'ManufacturerModelName'),                     ('Station', 'StationName'), ('Institution', 'InstitutionName'),                     ('SeriesDescription', 'SeriesDescription')):        row[key] = str(getattr(h, tag, '') or '')    for key, tag in (('FieldStrength', 'MagneticFieldStrength'),                     ('SliceThickness', 'SliceThickness')):        try:            row[key] = float(getattr(h, tag, None))        except (TypeError, ValueError):            row[key] = None    try:        row['Rows'], row['Columns'] = int(h.Rows), int(h.Columns)    except Exception:        pass    ps = getattr(h, 'PixelSpacing', None)    if ps is not None:        try:            row['ps_row'], row['ps_col'] = float(ps[0]), float(ps[1])        except Exception:            pass    return rowt0 = time.time()with ThreadPoolExecutor(max_workers=16) as ex:    rows = list(ex.map(read_one, tasks))print(f'{len(rows):,} headers in {(time.time() - t0) / 60:.1f} min')hdr = pd.DataFrame(rows)hdr['fov_mm'] = hdr.get('ps_row') * hdr.get('Rows')hdr.to_csv('/kaggle/working/series_headers.csv', index=False)n_err = hdr['error'].notna().sum() if 'error' in hdr else 0print(f'{n_err} failed reads')print(f'wrote /kaggle/working/series_headers.csv  {hdr.shape}')hdr.head(3)

## 5. Field of view and effective pitchThis is the number Section 8 currently says it cannot give.A one-millimetre feature survives resampling only at a pitch of 0.5 mm or less.The cache resized each series to 224 px with no physical crop, so the pitch isthe acquired field of view divided by 224. The last loop shows what resolutionwould have been needed instead - the top solutions used 384 px with a crop.

In [ ]:
import numpy as npok = hdr.dropna(subset=['fov_mm'])print(f'series with usable geometry: {len(ok):,} of {len(hdr):,}\n')pct = [.01, .05, .25, .5, .75, .95, .99]print('acquired field of view, mm')print(ok['fov_mm'].describe(percentiles=pct).round(1).to_string())pitch = ok['fov_mm'] / 224.0print('\neffective pitch after resize to 224 px, mm per pixel')print(pitch.describe(percentiles=pct).round(3).to_string())BOUND = 0.5print(f'\nmeeting the {BOUND} mm bound at 224 px: {100 * (pitch <= BOUND).mean():.2f}%')print(f'above it:                        {100 * (pitch > BOUND).mean():.2f}%')print('\nwhat other resize targets would have given:')for P in (224, 288, 336, 384, 448):    p = ok['fov_mm'] / P    print(f'  {P:>3} px   median {p.median():.3f} mm/px   {100 * (p <= BOUND).mean():5.1f}% meet the bound')print('\nwith a 140 mm physical crop, as the top solutions used:')for P in (224, 288, 384):    print(f'  {P:>3} px   {140 / P:.3f} mm/px   {"meets" if 140 / P <= BOUND else "fails"} the bound')

## 6. Scanner signatures as site clustersScanner settings identify the hospital, so a model trained with randomlystratified folds can learn a site's base rates instead of the knee. Thefifth-place team measured 0.019 of local AUC inflation from exactly that.Each study gets the modal signature across its series. Use this column as the`groups` argument to `GroupKFold` when re-running `tune_head.py`.

In [ ]:
parts = [hdr['Manufacturer'].fillna(''), hdr['Model'].fillna(''),         hdr['FieldStrength'].astype('string').fillna('')]hdr['scanner'] = parts[0].str.strip() + ' | ' + parts[1].str.strip() + ' | ' + parts[2]per_study = (hdr.groupby('StudyInstanceUID')['scanner']                .agg(lambda s: s.value_counts().index[0])                .rename('scanner')                .reset_index())per_study.to_csv('/kaggle/working/study_sites.csv', index=False)vc = per_study['scanner'].value_counts()print(f'{len(vc)} distinct scanner signatures over {len(per_study):,} studies\n')print(vc.head(25).to_string())print(f'\nsignatures with >= 30 studies: {(vc >= 30).sum()}, covering {vc[vc >= 30].sum():,} studies')print(f'singleton signatures: {(vc == 1).sum()}')print('\nwrote /kaggle/working/study_sites.csv')

## 7. What to do with the two filesCommit this notebook (Save Version -> Save & Run All) so `/kaggle/working`persists, then download both CSVs from the Output tab.- `series_headers.csv` gives the field-of-view histogram for Section 8. Send me  the printed summary from cell 5 and I will rewrite that paragraph.- `study_sites.csv` feeds the grouped-fold re-run. The next step is swapping  `StratifiedKFold` for `StratifiedGroupKFold` in `tune_head.py`, keyed on the  `scanner` column, and re-measuring the twelve probes. The gap between that  number and the current 0.775 is the leakage, and it is a reportable result  rather than only a correction.If `Manufacturer` and `ManufacturerModelName` come back mostly empty, fall backto `Station`, `Institution` or a rounded `SliceThickness` and `ps_row` pair -anything that varies by site and not by patient.